```text
Compared with C2, there are now two changes:

Dataset filtering: keep only adult → child examples.

LoRA targets: add GPT-2 attention layers:

attn.c_attn
attn.c_proj

in addition to:

mlp.c_fc
mlp.c_proj

The resulting architecture is:

GPT-2
│
├── Block 0
│   ├── Attention
│   │   ├── c_attn ← LoRA
│   │   └── c_proj ← LoRA
│   └── MLP
│       ├── c_fc   ← LoRA
│       └── c_proj ← LoRA
│
├── Block 1
│   └── ...
│
...
│
└── Block 11
    ├── Attention ← LoRA
    └── MLP       ← LoRA

```

In [1]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model


In [2]:
# ============================================================
# Configuration
# ============================================================

MODEL_NAME = "openai-community/gpt2"
DATASET_NAME = "nshah-fbcs/childes-engUK-conversational-pairs"

OUTPUT_DIR = "./gpt2-childes-lora-c3"

MAX_LENGTH = 1024

TRAIN_SIZE = 1000 #10_000
VAL_SIZE = 1_000
TEST_SIZE = 1_000

SEED = 42


# LoRA
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05


# Training
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01
BATCH_SIZE = 4
NUM_EPOCHS = 1

In [3]:
# ============================================================
# Load tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT-2 has no dedicated padding token.
tokenizer.pad_token = tokenizer.eos_token


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [4]:
# ============================================================
# Load dataset
# ============================================================

dataset = load_dataset(DATASET_NAME)

print(dataset)



README.md:   0%|          | 0.00/5.01k [00:00<?, ?B/s]

train.jsonl: reconstructing file:   0%|          |  0.00B / 34.1MB            

train.jsonl: downloading bytes:           |  0.00B            

validation.jsonl:   0%|          | 0.00/1.79M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/250757 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13197 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/14504 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['speaker_a', 'utterance_a', 'speaker_b', 'utterance_b'],
        num_rows: 250757
    })
    validation: Dataset({
        features: ['speaker_a', 'utterance_a', 'speaker_b', 'utterance_b'],
        num_rows: 13197
    })
    test: Dataset({
        features: ['speaker_a', 'utterance_a', 'speaker_b', 'utterance_b'],
        num_rows: 14504
    })
})


In [5]:

# ============================================================
# Keep only Adult -> Child examples
# ============================================================

def is_adult_to_child(example):
    speaker_a = str(example["speaker_a"]).strip().lower()
    speaker_b = str(example["speaker_b"]).strip().lower()

    return speaker_a == "adult" and speaker_b == "child"


dataset = dataset.filter(is_adult_to_child)

print("\nAfter adult -> child filtering:")
print(dataset)



Filter:   0%|          | 0/250757 [00:00<?, ? examples/s]

Filter:   0%|          | 0/13197 [00:00<?, ? examples/s]

Filter:   0%|          | 0/14504 [00:00<?, ? examples/s]


After adult -> child filtering:
DatasetDict({
    train: Dataset({
        features: ['speaker_a', 'utterance_a', 'speaker_b', 'utterance_b'],
        num_rows: 122113
    })
    validation: Dataset({
        features: ['speaker_a', 'utterance_a', 'speaker_b', 'utterance_b'],
        num_rows: 6494
    })
    test: Dataset({
        features: ['speaker_a', 'utterance_a', 'speaker_b', 'utterance_b'],
        num_rows: 7139
    })
})


In [6]:
# ============================================================
# Shuffle and create train/validation/test subsets
# ============================================================

all_data = dataset["train"].shuffle(seed=SEED)

required_size = TRAIN_SIZE + VAL_SIZE + TEST_SIZE

if len(all_data) < required_size:
    raise ValueError(
        f"Not enough adult->child examples. "
        f"Need {required_size}, but only {len(all_data)} are available."
    )

train_dataset = all_data.select(range(0, TRAIN_SIZE))

val_dataset = all_data.select(
    range(TRAIN_SIZE, TRAIN_SIZE + VAL_SIZE)
)


test_dataset = all_data.select(
    range(
        TRAIN_SIZE + VAL_SIZE,
        TRAIN_SIZE + VAL_SIZE + TEST_SIZE,
    )
)

print("\nDataset sizes:")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))


Dataset sizes:
Train: 1000
Validation: 1000
Test: 1000


In [7]:
# ============================================================
# Response-only SFT tokenization
# ============================================================

def tokenize_example(example):
    adult_text = example["utterance_a"]
    child_text = example["utterance_b"]

    prompt = f"Adult: {adult_text}\nChild:"

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
    )["input_ids"]

    response_ids = tokenizer(
        " " + child_text,
        add_special_tokens=False,
    )["input_ids"]

    # Explicit EOS target.
    response_ids = response_ids + [tokenizer.eos_token_id]


    input_ids = prompt_ids + response_ids

    # Ignore the prompt when computing loss.
    labels = (
        [-100] * len(prompt_ids)
        + response_ids
    )

    # Truncate both together.
    input_ids = input_ids[:MAX_LENGTH]
    labels = labels[:MAX_LENGTH]

    return {
        "input_ids": input_ids,
        "labels": labels,
    }


train_dataset = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names,
)

val_dataset = val_dataset.map(
    tokenize_example,
    remove_columns=val_dataset.column_names,
)

test_dataset = test_dataset.map(
    tokenize_example,
    remove_columns=test_dataset.column_names,
)



Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [8]:
# ============================================================
# Dynamic padding collator
# ============================================================

def collate_fn(features):
    input_ids = [
        item["input_ids"]
        for item in features
    ]

    labels = [
        item["labels"]
        for item in features
    ]

    batch_input_ids = tokenizer.pad(
        {
            "input_ids": input_ids,
        },
        padding=True,
        return_tensors="pt",
    )["input_ids"]


    # Labels use -100 for padding so PyTorch CrossEntropyLoss
    # ignores those positions.
    max_length = batch_input_ids.shape[1]

    padded_labels = []

    for label in labels:
        padded = label + [-100] * (max_length - len(label))
        padded_labels.append(padded)

    import torch

    batch_labels = torch.tensor(
        padded_labels,
        dtype=torch.long,
    )

    attention_mask = (
        batch_input_ids != tokenizer.pad_token_id
    ).long()


    return {
        "input_ids": batch_input_ids,
        "labels": batch_labels,
        "attention_mask": attention_mask,
    }



In [9]:
# ============================================================
# Load GPT-2
# ============================================================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
)

model.config.pad_token_id = tokenizer.pad_token_id


model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [10]:
# ============================================================
# C3 LoRA target modules
#
# GPT-2 block:
#
#   attention:
#       c_attn
#       c_proj
#
#   MLP:
#       c_fc
#       c_proj
#
# Apply LoRA to all four in all 12 blocks.
# ============================================================

target_modules = []


for block_idx in range(12):
    target_modules.append(
        f"transformer.h.{block_idx}.attn.c_attn"
    )

    target_modules.append(
        f"transformer.h.{block_idx}.attn.c_proj"
    )

    target_modules.append(
        f"transformer.h.{block_idx}.mlp.c_fc"
    )

    target_modules.append(
        f"transformer.h.{block_idx}.mlp.c_proj"
    )


print("\nLoRA target modules:")
for module in target_modules:
    print(" ", module)




LoRA target modules:
  transformer.h.0.attn.c_attn
  transformer.h.0.attn.c_proj
  transformer.h.0.mlp.c_fc
  transformer.h.0.mlp.c_proj
  transformer.h.1.attn.c_attn
  transformer.h.1.attn.c_proj
  transformer.h.1.mlp.c_fc
  transformer.h.1.mlp.c_proj
  transformer.h.2.attn.c_attn
  transformer.h.2.attn.c_proj
  transformer.h.2.mlp.c_fc
  transformer.h.2.mlp.c_proj
  transformer.h.3.attn.c_attn
  transformer.h.3.attn.c_proj
  transformer.h.3.mlp.c_fc
  transformer.h.3.mlp.c_proj
  transformer.h.4.attn.c_attn
  transformer.h.4.attn.c_proj
  transformer.h.4.mlp.c_fc
  transformer.h.4.mlp.c_proj
  transformer.h.5.attn.c_attn
  transformer.h.5.attn.c_proj
  transformer.h.5.mlp.c_fc
  transformer.h.5.mlp.c_proj
  transformer.h.6.attn.c_attn
  transformer.h.6.attn.c_proj
  transformer.h.6.mlp.c_fc
  transformer.h.6.mlp.c_proj
  transformer.h.7.attn.c_attn
  transformer.h.7.attn.c_proj
  transformer.h.7.mlp.c_fc
  transformer.h.7.mlp.c_proj
  transformer.h.8.attn.c_attn
  transformer.h.8.at

In [11]:
# ============================================================
# LoRA configuration
# ============================================================

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=target_modules,
)



In [14]:
"""
/usr/local/lib/python3.13/dist-packages/peft/import_utils.py in is_torchao_available()
    160
    161     if torchao_version < TORCHAO_MINIMUM_VERSION:
--> 162         raise ImportError(
    163             f"Found an incompatible version of torchao. Found version {torchao_version}, "
    164             f"but only versions above {TORCHAO_MINIMUM_VERSION} are supported"

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported
"""
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 48.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [15]:

# ============================================================
# Add LoRA adapters
# ============================================================

model = get_peft_model(
    model,
    lora_config,
)

model.print_trainable_parameters()



trainable params: 1,179,648 || all params: 125,619,456 || trainable%: 0.9391


/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2760: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [16]:
# ============================================================
# Sanity checks
# ============================================================

trainable_names = [
    name
    for name, param in model.named_parameters()
    if param.requires_grad
]

# Every trainable parameter should be a LoRA parameter.
assert all(
    "lora_" in name
    for name in trainable_names
)

# All trainable parameters should belong to attention or MLP.
assert all(
    ".attn." in name or ".mlp." in name
    for name in trainable_names
)


# We expect all four target module types.
assert any(".attn.c_attn." in name for name in trainable_names)
assert any(".attn.c_proj." in name for name in trainable_names)
assert any(".mlp.c_fc." in name for name in trainable_names)
assert any(".mlp.c_proj." in name for name in trainable_names)



In [18]:
# ============================================================
# Training arguments
# ============================================================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=100,

    report_to="none",


    # Keep this simple on the Intel Mac.
    fp16=False,
    bf16=False,

    seed=SEED,
)



In [20]:
# ============================================================
# Trainer
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    processing_class=tokenizer,
    data_collator=collate_fn,
)



In [21]:

# ============================================================
# Train
# ============================================================

trainer.train()



[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.383631,3.221265


TrainOutput(global_step=250, training_loss=3.4314241333007813, metrics={'train_runtime': 21.7635, 'train_samples_per_second': 45.949, 'train_steps_per_second': 11.487, 'total_flos': 13225098977280.0, 'train_loss': 3.4314241333007813, 'epoch': 1.0})

In [22]:
# ============================================================
# Evaluate on validation set
# ============================================================

print("\nValidation evaluation:")
val_metrics = trainer.evaluate()
print(val_metrics)



Validation evaluation:


Training Loss,Validation Loss,Epoch
3.383631,3.221265,1


{'eval_loss': 3.221264600753784}


In [23]:
# ============================================================
# Evaluate on held-out test set
# ============================================================

print("\nTest evaluation:")
test_metrics = trainer.evaluate(
    eval_dataset=test_dataset
)
print(test_metrics)



Test evaluation:


Training Loss,Validation Loss,Epoch
3.383631,3.208872,1


{'eval_loss': 3.208872079849243}


In [24]:
# ============================================================
# Save LoRA adapter
# ============================================================

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)



('./gpt2-childes-lora-c3/tokenizer_config.json',
 './gpt2-childes-lora-c3/tokenizer.json')

In [25]:
# ============================================================
# Generation helper
# ============================================================

def generate_child_response(adult_text):
    prompt = f"Adult: {adult_text}\nChild:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    )

    prompt_length = inputs["input_ids"].shape[1]

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=50,

        do_sample=True,
        temperature=0.8,
        top_p=0.9,


        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

    # Remove the prompt from the decoded output.
    new_token_ids = generated_ids[
        0,
        prompt_length:,
    ]

    response = tokenizer.decode(
        new_token_ids,
        skip_special_tokens=False,
    )

    return response


In [26]:
# ============================================================
# Example
# ============================================================

print("\nExample generation:")
print(
    generate_child_response(
        "What are you doing?"
    )
)



Example generation:


/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:2944: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


 i'm doing the dog's tail dance .<|endoftext|>


In [27]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [28]:
!jupyter nbconvert --to html /content/drive/MyDrive/"Colab Notebooks"/sft_childes_gpt2_lora_c3.ipynb --output-dir=/content/drive/MyDrive/

[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/sft_childes_gpt2_lora_c3.ipynb to html
[NbConvertApp] Writing 549710 bytes to /content/drive/MyDrive/sft_childes_gpt2_lora_c3.html
